# 强化学习与大模型后训练 · 第 4/12 课：GAE 与优势估计

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现带终止 mask 的 GAE 反向递推，并解释 λ 如何连接 TD 与 MC。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、概率期望、基本深度学习
- 本课在路线中的作用：GAE 把多个 n-step TD error 以 (γλ)^k 加权，控制优势估计的偏差—方差。

## 核心心智模型

### 1. 它是什么，解决什么问题

GAE 把多个 n-step TD error 以 (γλ)^k 加权，控制优势估计的偏差—方差。

### 2. 它如何工作

从后向前递推 A_t=δ_t+γλ(1-done_t)A_{t+1}，其中 δ_t=r_t+γ(1-done_t)V_{t+1}-V_t。

### 3. 正确性条件与常见误区

values 必须比 rewards 多一个 bootstrap 值；done mask 应同时作用于 δ 的 bootstrap 和递推尾项。

### 4. 性能与工程取舍

λ→0 接近 TD(0)，λ→1 接近 Monte Carlo；长 horizon 下 λ 大会放大噪声。

## 具体演示

两步轨迹可先算 δ₁，再算 A₀；这比背整串加权和更不易错。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 GAE 中两个 mask 位置。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
def gae(rewards, values, dones, gamma, lam):
    assert len(values) == len(rewards) + 1
    adv = [0.0] * len(rewards)
    carry = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        not_done = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * ______ * values[t + 1] - values[t]
        carry = delta + gamma * lam * ______ * carry
        adv[t] = carry
    return adv

got = gae([1., 1.], [0.5, 0.6, 0.0], [False, True], .9, .95)
assert all(abs(a-b) < 1e-9 for a,b in zip(got, [1.382, 0.4]))


### 检查方法

运行断言，并解释最后一步为何不读取 `values[2]` 的数值贡献。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“GAE 与优势估计”的工作机制。

**你的答案：**


### Q2

只在 δ 上乘 done mask、递推 carry 不乘，会跨 episode 泄漏什么？

**你的答案：**


### Q3

LLM token 级奖励只有末尾一个分数时，GAE 的信用会如何向前传播？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [GAE paper](https://arxiv.org/abs/1506.02438)

资料用于建立事实基线；面试回答仍需用自己的语言组织。